# Analyzing Code Performance

### Timing Your Code

In [3]:
def slow_way_to_calculate_mode(list_of_numbers):
    result_dict = {}
    for i in list_of_numbers:
        if i not in result_dict:
            result_dict[i] = 1
        else:
            result_dict[i] += 1

    mode_vals = []
    max_frequency = max(result_dict.values())
    for key, value in result_dict.items():
        if value == max_frequency:
            mode_vals.append(key)

    return mode_vals

In [4]:
slow_way_to_calculate_mode([4, 5, 5, 6])

[5]

In [5]:
import numpy as np

random_integers = np.random.randint(1, 1_000_000, 1_000_000)

In [6]:
import time

start = time.time()
slow_way_to_calculate_mode(random_integers)
end = time.time()

print(end - start)

0.3450000286102295


In [7]:
%%timeit
slow_way_to_calculate_mode(random_integers)

277 ms ± 17.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Profiling Your Code

### `cProfile`

In [8]:
from collections import Counter
import numpy as np

In [9]:
def mode_using_counter(n_integers):
    random_integers = np.random.randint(1, 100000, n_integers)
    c = Counter(random_integers)
    return c.most_common(1)[0][0]

In [10]:
mode_using_counter(10000000)

78504

In [11]:
%%timeit
mode_using_counter(10000000)

1.38 s ± 55.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%prun
mode_using_counter(10000000)

         23 function calls in 1.351 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    1.236    1.236    1.236    1.236 {built-in method _collections._count_elements}
        1    0.110    0.110    0.110    0.110 {method 'randint' of 'numpy.random.mtrand.RandomState' objects}
        1    0.002    0.002    0.002    0.002 {built-in method builtins.max}
        1    0.001    0.001    1.351    1.351 <string>:1(<module>)
        1    0.001    0.001    1.237    1.237 __init__.py:640(update)
        1    0.000    0.000    0.000    0.000 {method 'reduce' of 'numpy.ufunc' objects}
        1    0.000    0.000    1.349    1.349 3568474488.py:1(mode_using_counter)
        1    0.000    0.000    1.351    1.351 {built-in method builtins.exec}
        1    0.000    0.000    0.000    0.000 abc.py:117(__instancecheck__)
        1    0.000    0.000    0.000    0.000 {built-in method builtins.iter}
        1    0.000    0.000    1.

In [16]:
%load_ext snakeviz

The snakeviz extension is already loaded. To reload it, use:
  %reload_ext snakeviz


In [17]:
%%snakeviz
mode_using_counter(1000000)

 
*** Profile stats marshalled to file '/tmp/tmp6usdkuv7'.
Embedding SnakeViz in this document...
<function display at 0xff79dabcae60>


`line_profiler`

In [23]:
%load_ext line_profiler

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


In [24]:
%lprun -f mode_using_counter mode_using_counter(10000000)

Timer unit: 1e-09 s

Total time: 1.36075 s
File: /tmp/ipykernel_3287/3568474488.py
Function: mode_using_counter at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def mode_using_counter(n_integers):
     2         1  130290033.0    1e+08      9.6      random_integers = np.random.randint(1, 100000, n_integers)
     3         1 1227976303.0    1e+09     90.2      c = Counter(random_integers)
     4         1    2481089.0    2e+06      0.2      return c.most_common(1)[0][0]

## Time Complexity

In [ ]:
def weighted_mean(list_of_numbers, weights):
    running_total = 0
    for i in range(len(list_of_numbers)):
        running_total += (list_of_numbers[i] * weights[i])
    return (running_total/sum(weights))

In [ ]:
def covariance_fast(X, Y):
    avg_X = sum(X) / len(X)
    avg_Y = sum(Y) / len(Y)

    result = 0
    for i in range(len(X)):
        result += (X[i] - avg_X) * (Y[i] - avg_Y)

    return result / len(X)

In [ ]:
X = np.random.randint(1, 1000, 1000)
Y = np.random.randint(1, 1000, 1000)

In [ ]:
%%timeit
covariance_fast(X, Y)

In [ ]:
def covariance(X, Y):
    cov_sum = 0
    for i in range(len(X)):
        for j in range(len(Y)):
            cov_sum += 0.5 * (X[i] - X[j]) * (Y[i] - Y[j])
    return cov_sum / (len(X) ** 2)

In [ ]:
%%timeit
covariance(X, Y)

## Code to generate Figure 2.3

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
n = np.linspace(1, 10, 1000)
line_names = [
    "Constant",
    "Linear",
    "Quadratic",
    "Exponential",
    "Logarithmic",
    "N log N",
]

colors = ["black", "orange", "green", "blue", "red", [0.5, 0.5, 0.5]]
linestyles = ["solid", "solid", (0, (5, 5)), "dotted", "dashdot", (0, (5, 1))]
big_o = [np.ones(n.shape), n, n**2, 2**n, np.log(n), n * (np.log(n))]

fig, ax = plt.subplots()
fig.set_facecolor("white")

ax.set_ylim(0, 50)
ax.set_xlim(1, 10)
for i in range(len(big_o)):
    ax.plot(n, big_o[i], label=line_names[i], color=colors[i], linestyle=linestyles[i])
ax.set_ylabel("Relative Runtime")
ax.set_xlabel("Input Size")
ax.legend()
fig.savefig("seds_0202_v3.png", bbox_inches="tight")